# G-V calibration from NASA IWG1 in-situ logs

Mirrors the G-III calibration in
`notebooks/calibration/giii/calibration.ipynb`, adapted to the
NASA 95 G-V data delivered as one or more concatenated
`n95*_alltracks.csv` files (split into per-sortie files via
`hyplan.aircraft.split_iwg1_alltracks`).

The output of this notebook is a paste-ready `NASA_GV()`
constructor block (§11).  The G-V is a long-range business jet
with a service ceiling of FL510 and typical cruise FL410-FL490,
so the altitude bins extend higher than the G-III but the
methodology is the same.


In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hyplan import ureg
from hyplan.aircraft import load_iwg1, trim_ground_taxi, NASA_GV

# Shared helpers live one directory up (notebooks/calibration/_common.py).
sys.path.insert(0, str(Path("..").resolve()))
from _common import (
    label_phases, apply_sortie_filters, per_bin, tas_per_bin,
    schedule_pts, evaluate_profile, summary_table,
)

DATA_DIR = Path("../../../data/gv").resolve()
TAIL_GLOB = "n95_*.txt"

# Phase-label thresholds.  Same defaults as the ER-2 notebook —
# climb/descent gate at 300 fpm separates sustained vertical motion
# from autopilot ±100 ft cruise oscillation.
CLIMB_FPM   = 300.0
DESCENT_FPM = -300.0

# Sortie filters: real flights only.
MIN_DUR_MIN     = 60.0   # below this is taxi / engine run
MAX_DUR_MIN     = 900.0  # G-V long-range; up to ~13 hr
MIN_PEAK_ALT_FT = 30000  # G-V cruise floor; below FL300 is test/ferry


## 1. Load + trim ground taxi + phase-label every sortie


In [2]:
sorties = {}
skipped = []
for p in sorted(DATA_DIR.glob(TAIL_GLOB)):
    raw = load_iwg1(p)
    a = trim_ground_taxi(raw)
    a, reason = apply_sortie_filters(
        a, min_dur_min=MIN_DUR_MIN, max_dur_min=MAX_DUR_MIN,
        min_peak_alt_ft=MIN_PEAK_ALT_FT,
    )
    if reason is not None:
        skipped.append((p.stem, reason))
        continue
    sorties[p.stem] = label_phases(a, climb_fpm=CLIMB_FPM, descent_fpm=DESCENT_FPM)

summary_table(sorties, skipped, source_label=str(DATA_DIR.name))


source:            gv
raw files:         253
valid sorties:     101
excluded:          152
excluded by reason:
    70  no airborne fixes
    52  low peak alt
    26  no valid altitude
     4  too short
date range:        2019-05-18 → 2026-04-11


,source,raw_files,valid_sorties,excluded,exclusion_reasons,date_range
0,gv,253,101,152,"70 no airborne fixes, 52 low peak alt, 26 no v...",2019-05-18 → 2026-04-11


## 2. Per-sortie altitude profiles


In [3]:
fig, ax = plt.subplots(figsize=(13, 5))
for name, a in sorties.items():
    t_min = (a["timestamp"] - a["timestamp"].iloc[0]).dt.total_seconds() / 60.0
    ax.plot(t_min, a["altitude"] / 1000, lw=0.6, alpha=0.4, color="steelblue")
ax.set_xlabel("minutes from takeoff")
ax.set_ylabel("altitude (kft)")
ax.set_title(f"NASA 95 G-V altitude profiles — {len(sorties)} sorties")
ax.grid(alpha=0.3)
ax.set_ylim(0, 50)
plt.tight_layout()
plt.show()


/var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/ipykernel_99422/1671502133.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Ground tracks


In [4]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(13, 7))
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor="0.95")
ax.add_feature(cfeature.OCEAN, facecolor="0.85")
ax.add_feature(cfeature.COASTLINE, lw=0.4)
ax.add_feature(cfeature.BORDERS, lw=0.4, ls=":")

for name, a in sorties.items():
    ax.plot(a["longitude"], a["latitude"], lw=0.4, alpha=0.4,
            color="steelblue", transform=ccrs.PlateCarree())

# Bound to the sortie footprint.
all_lat = pd.concat([a["latitude"] for a in sorties.values()])
all_lon = pd.concat([a["longitude"] for a in sorties.values()])
pad = 2
ax.set_extent([all_lon.min()-pad, all_lon.max()+pad,
               all_lat.min()-pad, all_lat.max()+pad])
ax.set_title(f"NASA 95 G-V ground tracks — {len(sorties)} sorties")
plt.tight_layout()
plt.show()


/var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/ipykernel_99422/2434519477.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Climb / descent: per-altitude-bin medians

Same active-climb / active-descent definitions used for ER-2 phase 2:
restrict to fixes with VS > 1500 fpm (or < -1500 fpm for descent) so
the per-bin medians represent pure aircraft performance, not cruise
plateaus or weight-management holds the planner should model
separately via `typical_climb_out` if needed.


In [5]:
ACTIVE_VS_THR_FPM = 1500.0
BIN_FT = 5000

climb_bins   = per_bin(sorties, "climb",   +1, ACTIVE_VS_THR_FPM, bin_ft=BIN_FT,
                       extra_cols=("tas_kt", "mach"))
descent_bins = per_bin(sorties, "descent", -1, ACTIVE_VS_THR_FPM, bin_ft=BIN_FT,
                       extra_cols=("tas_kt", "mach"))

print("Active CLIMB (VS >= 1500 fpm):")
print(climb_bins.to_string(index=False))
print()
print("Active DESCENT (VS <= -1500 fpm):")
print(descent_bins.to_string(index=False))


Active CLIMB (VS >= 1500 fpm):
 alt_bin_ft    n  vs_med  vs_p25  vs_p75  tas_med  mach_med
          0 1566  2201.0  1838.7  2513.2    234.0       0.4
       5000 1758  2544.2  2238.1  2775.9    279.3       0.4
      10000 2262  2280.3  2087.5  2502.3    352.2       0.6
      15000 2269  2278.0  2042.3  2482.5    385.7       0.6
      20000 2282  2020.3  1820.7  2294.5    417.3       0.7
      25000 2141  1911.3  1775.2  2137.0    444.0       0.7
      30000 1568  1818.5  1657.4  2008.1    443.2       0.8
      35000  490  1733.5  1594.9  1902.4    439.7       0.8

Active DESCENT (VS <= -1500 fpm):
 alt_bin_ft    n  vs_med  vs_p25  vs_p75  tas_med  mach_med
          0  164 -1703.4 -1910.4 -1602.1    270.0       0.4
       5000 1278 -1805.8 -2066.8 -1592.8    284.2       0.4
      10000 1788 -1872.9 -2230.2 -1637.3    341.7       0.5
      15000 1972 -2198.7 -2521.9 -1931.5    383.1       0.6
      20000 2131 -2305.7 -2529.0 -2011.3    410.2       0.7
      25000 1904 -2425.2 -2750.1 -

## 5. Climb_profile breakpoints

VerticalProfile points pinned at:

* SL up to the highest populated bin: per 5-kft bin, the active-climb
  median from §4 (n>=30/bin).
* FL300 (certified ceiling, brochure): residual rate so the integrator
  terminates cleanly if a planner asks for ceiling.


In [6]:
CEILING_FT     = 51000           # G-V certified ceiling
# Residual climb rate at the certified ceiling — the FL410 active-climb
# bin shows ~1100 fpm and the FL400 cruise-altitude bin sees no
# active-climb fixes (n=0), so we extrapolate to a small positive rate
# at the certified ceiling.  Not a regulatory definition (FAR Part 25
# service ceiling is 500 fpm at MTOW); just an anchor that lets the
# integration terminate if a planner asks for 45 kft.
CEILING_VS_FPM = 500.0

# Use the active-climb medians directly.  Don't enforce monotone-
# decreasing-from-SL: jets typically peak ROC near FL050-100 (limited
# below by 250-KCAS ATC procedures), so a clamp would push bins below
# their IQRs.  If the resulting profile shows wobble (small-sample
# bins), that's a separate decision and shouldn't be hidden in the
# calibration step.
fixed = []
for _, r in climb_bins.iterrows():
    if r["alt_bin_ft"] >= 0 and r["alt_bin_ft"] < CEILING_FT:
        fixed.append((int(r["alt_bin_ft"]), float(r["vs_med"])))
fixed.append((CEILING_FT, CEILING_VS_FPM))

print("climb_profile points (alt_ft, vs_fpm):")
for alt, vs in fixed:
    print(f"  ({alt:>5d}, {vs:6.0f})")


climb_profile points (alt_ft, vs_fpm):
  (    0,   2201)
  ( 5000,   2544)
  (10000,   2280)
  (15000,   2278)
  (20000,   2020)
  (25000,   1911)
  (30000,   1818)
  (35000,   1734)
  (51000,    500)


## 6. Descent_profile breakpoints

Same construction: per-bin medians of active descent VS, anchored at
top of approach (~300-500 ft AGL) and at cruise altitude.  The
planner will steepen this to fit short legs via
`descent_path_angle_max_deg=6.0` (same descent-shortening posture used by the calibrated ER-2 model).


In [7]:
# Use the active-descent medians directly.  Don't enforce monotone-
# increasing-with-altitude: descent VS typically peaks around FL150-200
# (where the aircraft descends near VMO in CAS), then declines in the
# upper levels (Mach-limited descent at constant M).  The previous
# monotone-from-SL clamp was forcing every bin above FL150 above its
# IQR and obscuring the real shape.
fixed_desc = []
for _, r in descent_bins.iterrows():
    if 0 <= r["alt_bin_ft"] < 45000:
        fixed_desc.append((int(r["alt_bin_ft"]), abs(float(r["vs_med"]))))
fixed_desc = sorted(fixed_desc)

print("descent_profile points (alt_ft, |vs|_fpm):")
for alt, vs in fixed_desc:
    print(f"  ({alt:>5d}, {vs:6.0f})")


descent_profile points (alt_ft, |vs|_fpm):
  (    0,   1703)
  ( 5000,   1806)
  (10000,   1873)
  (15000,   2199)
  (20000,   2306)
  (25000,   2425)
  (30000,   2432)
  (35000,   2091)
  (40000,   1824)


## 7. TAS schedules: climb / cruise / descent

Three independent TAS-vs-altitude schedules, derived from per-phase
medians of the same IWG1 fixes that drove the climb_profile and
descent_profile.  At the same altitude the three phases differ
materially — at FL300 cruise TAS is +39 kt over climb; at FL400 the
descent is +26 kt over climb — so a single schedule shared across
phases (or a fixed-offset derivation like
`_descent_schedule_from_cruise(cruise, 49)`) under-reads cruise TAS
during cruise and mis-models descent.


In [8]:
climb_tas   = tas_per_bin(sorties, ["climb"],   bin_ft=BIN_FT, n_min=50)
cruise_tas  = tas_per_bin(sorties, ["cruise"],  bin_ft=BIN_FT, n_min=50)
descent_tas = tas_per_bin(sorties, ["descent"], bin_ft=BIN_FT, n_min=50)

print("Climb TAS:")
print(climb_tas.to_string(index=False))
print()
print("Cruise TAS:")
print(cruise_tas.to_string(index=False))
print()
print("Descent TAS:")
print(descent_tas.to_string(index=False))


# Climb: SL rotation -> ceiling, climb-phase medians.  Anchor SL at
# typical jet rotation TAS (~150 kt) since the SL climb-phase bin is
# contaminated by takeoff-roll fixes still accelerating.
ROTATION_TAS_KT = 150
climb_pts = [(0, ROTATION_TAS_KT)] + schedule_pts(
    climb_tas, [10000, 20000, 30000, 40000, 45000, 50000], n_min=200
)

# Cruise: G-V typical band FL410-FL510 with step climbs.  Below
# FL410 cruise-labeled bins are mostly transient level-offs.
cruise_pts = schedule_pts(cruise_tas, [41000, 45000, 47000, 49000, 51000], n_min=200)

# Descent: low-altitude anchor + descent-phase medians up to
# cruise ceiling.
descent_pts = schedule_pts(descent_tas, [10000, 20000, 30000, 40000, 45000, 50000], n_min=200)
descent_pts = [(0, 150)] + descent_pts

print()
print("Climb schedule  (alt_ft, tas_kt):", climb_pts)
print("Cruise schedule (alt_ft, tas_kt):", cruise_pts)
print("Descent schedule(alt_ft, tas_kt):", descent_pts)



Climb TAS:
 alt_bin_ft    n  tas_med  mach_med
      -5000  300    114.2       0.2
          0 3680    233.8       0.4
       5000 2681    277.4       0.4
      10000 2704    351.8       0.6
      15000 2464    386.4       0.6
      20000 2842    418.3       0.7
      25000 3061    445.5       0.7
      30000 3131    446.7       0.8
      35000 4332    439.3       0.8
      40000 1584    454.9       0.8

Cruise TAS:
 alt_bin_ft      n  tas_med  mach_med
      -5000   2447      0.0       0.0
          0  13120    243.2       0.4
       5000   9593    262.8       0.4
      10000   5194    279.9       0.5
      15000   3034    390.1       0.6
      20000   1925    368.4       0.6
      25000  13584    425.1       0.7
      30000  18445    467.5       0.8
      35000  42308    465.6       0.8
      40000 161999    455.2       0.8
      45000    890    444.9       0.8

Descent TAS:
 alt_bin_ft    n  tas_med  mach_med
      -5000   98    109.1       0.2
          0 7681    193.2       0.3
  

## 8. Bank-angle analysis

`max_bank_deg` sets the minimum turn radius the Dubins planner uses;
it should reflect the *operational maximum* the aircraft is willing
to use during survey-line transitions, not the typical-mix median.

The 40k turn-fix sample (|Roll| > 5°) is dominated by small in-cruise
course corrections, so its median understates the bank used during
real maneuvers.  Use p90 instead — it captures the operational
ceiling without touching the steep-turn / emergency envelope.


In [9]:
ROLL_GATE_DEG = 5.0
banks = []
for a in sorties.values():
    abs_roll = a["roll_deg"].abs()
    in_turn = abs_roll > ROLL_GATE_DEG
    banks.append(abs_roll[in_turn])
all_banks = pd.concat(banks).dropna()
print(f"n turn fixes: {len(all_banks):,}")
print(f"|Roll| median:  {all_banks.median():.1f}°")
print(f"|Roll| p75/p90: {all_banks.quantile(0.75):.1f}° / {all_banks.quantile(0.90):.1f}°")


n turn fixes: 42,978
|Roll| median:  17.0°
|Roll| p75/p90: 24.9° / 27.3°


## 9. Operational vs aircraft-intrinsic framing

The TOC, approach-speed, and per-sortie peak-altitude statistics
that follow describe **operational** behavior across this sortie
set: wall-clock time-to-FL410 includes pre-cruise level-offs, ATC
routing, and weight-management step climbs; per-sortie peaks
reflect actual mission profiles flown rather than the airframe
service ceiling under MTOW; approach TAS is the median final-
approach speed for the mission mix.  Aircraft-intrinsic
performance (climb / descent / cruise schedules in §5–§7, bank
in §8) is what the planner consumes; the §9 numbers are reviewer-
facing context.


## 9b. TOC, approach speed, service ceiling

Empirical observations to feed the SourceRecord and the
`approach_speed` / `service_ceiling` parameters.


In [10]:
# TOC = takeoff -> first FL410 fix (G-V typical initial cruise).
toc = []
for a in sorties.values():
    above = a[a["altitude"] >= 41000]
    if not above.empty:
        toc.append((above["timestamp"].iloc[0] - a["timestamp"].iloc[0]).total_seconds() / 60.0)
toc_s = pd.Series(toc)

# Approach speed: median TAS in the last 500 ft AGL with VS < -200 fpm.
# Tighten the window from the G-III's ±1500 ft so we get final-approach
# speed, not the wider pattern speed which inflates the median.
approach_tas = []
for a in sorties.values():
    floor = a["altitude"].min()
    sub = a[(a["altitude"] - floor < 500) & (a["altitude"] - floor > 50)
            & (a["vertical_rate"] < -200)]
    if not sub.empty:
        approach_tas.append(sub["tas_kt"].median())
ap_s = pd.Series(approach_tas).dropna()

# Service ceiling: p99 of per-sortie peak altitudes.
peaks = pd.Series([a["altitude"].max() for a in sorties.values()])

print(f"TOC (takeoff -> FL410):  n={len(toc_s)}, median {toc_s.median():.1f} min, IQR {toc_s.quantile(.25):.1f}-{toc_s.quantile(.75):.1f}, range {toc_s.min():.1f}-{toc_s.max():.1f}")
print(f"Approach TAS:            n={len(ap_s)}, median {ap_s.median():.0f} kt, IQR {ap_s.quantile(.25):.0f}-{ap_s.quantile(.75):.0f}")
print(f"Per-sortie peak alt:     median {peaks.median():.0f} ft, p99 {peaks.quantile(0.99):.0f} ft, max {peaks.max():.0f} ft")


TOC (takeoff -> FL410):  n=69, median 60.8 min, IQR 27.7-334.3, range 0.0-571.3
Approach TAS:            n=90, median 126 kt, IQR 122-131
Per-sortie peak alt:     median 41093 ft, p99 45321 ft, max 45426 ft


## 10. Validate calibrated profiles against observed data

Overlay the proposed VerticalProfile breakpoints on the per-bin
medians + IQR shading.  No comparison line against the shipping
`NASA_GV()` — every commit makes the two trivially identical, and
the IQR + medians alone tell the calibration story.


In [11]:
alt_grid = np.arange(0, 52000, 500)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.fill_betweenx(climb_bins["alt_bin_ft"]/1000,
                 climb_bins["vs_p25"], climb_bins["vs_p75"],
                 alpha=0.25, color="steelblue", label="active-climb IQR")
ax.plot(climb_bins["vs_med"], climb_bins["alt_bin_ft"]/1000,
        "o", color="steelblue", label="active-climb median")
ax.plot(evaluate_profile(fixed, alt_grid), alt_grid/1000,
        color="C1", lw=2, label="proposed")
ax.set_xlabel("VS (fpm)"); ax.set_ylabel("altitude (kft)")
ax.set_title("Climb profile"); ax.legend(loc="upper right")
ax.grid(alpha=0.3); ax.set_xlim(0, 6000)

ax = axes[1]
ax.fill_betweenx(descent_bins["alt_bin_ft"]/1000,
                 (-descent_bins["vs_p75"]).abs(),
                 (-descent_bins["vs_p25"]).abs(),
                 alpha=0.25, color="firebrick", label="active-descent IQR")
ax.plot((-descent_bins["vs_med"]).abs(), descent_bins["alt_bin_ft"]/1000,
        "o", color="firebrick", label="active-descent median")
ax.plot(evaluate_profile(fixed_desc, alt_grid), alt_grid/1000,
        color="C1", lw=2, label="proposed")
ax.set_xlabel("|VS| (fpm)"); ax.set_ylabel("altitude (kft)")
ax.set_title("Descent profile"); ax.legend(loc="upper right")
ax.grid(alpha=0.3); ax.set_xlim(0, 6000)

plt.tight_layout()
plt.show()


/var/folders/tk/dltx8gp544z3_ddzcb8c1_7r0000gn/T/ipykernel_99422/2132893233.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Code-paste-ready constructor block


In [12]:
print("# Paste into hyplan/aircraft/_models.py NASA_GV.__init__")
print()
print("climb_profile=VerticalProfile(points=[")
for alt, vs in fixed:
    print(f"    ({alt:>5d} * ureg.feet, {vs:6.0f} * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)")
print("]),")
print()
print("descent_profile=VerticalProfile(points=[")
for alt, vs in fixed_desc:
    print(f"    ({alt:>5d} * ureg.feet, {vs:6.0f} * ureg.feet / ureg.minute),  # active-descent median (n>=30/bin)")
print("]),")
print()
print("climb_schedule=TasSchedule(points=[")
for alt, tas in climb_pts:
    print(f"    ({alt:>5d} * ureg.feet, {tas:3d} * ureg.knot),  # climb-phase median")
print("]),")
print()
print("cruise_schedule=TasSchedule(points=[")
for alt, tas in cruise_pts:
    print(f"    ({alt:>5d} * ureg.feet, {tas:3d} * ureg.knot),  # cruise-phase median")
print("]),")
print()
print("descent_schedule=TasSchedule(points=[")
for alt, tas in descent_pts:
    print(f"    ({alt:>5d} * ureg.feet, {tas:3d} * ureg.knot),  # descent-phase median")
print("]),")
print()
print(f"# Approach speed: median TAS in last-500-ft AGL descent across {len(ap_s)} sorties.")
print(f"approach_speed={int(round(ap_s.median()))} * ureg.knot,")
print()
print(f"# Service ceiling: p99 of per-sortie peak altitude across {len(sorties)} sorties.")
print(f"service_ceiling={int(round(peaks.quantile(0.99) / 1000) * 1000)} * ureg.feet,")
print()
print(f"# p90 |Roll| during turns ({len(all_banks):,} fixes, gate >5°) —")
print(f"# operational maximum, not the typical-mix median ({all_banks.median():.1f}°).")
print(f"turn_model=TurnModel(max_bank_deg={int(round(all_banks.quantile(0.90)))}.0),")
print()
print(f'sources=[SourceRecord(')
print(f'    source_type="iwg1",')
print(f'    reference="NASA 95 IWG1 calibration, n={len(sorties)} sorties",')
print(f'    confidence=0.85,')
print(f')],')


# Paste into hyplan/aircraft/_models.py NASA_GV.__init__

climb_profile=VerticalProfile(points=[
    (    0 * ureg.feet,   2201 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    ( 5000 * ureg.feet,   2544 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (10000 * ureg.feet,   2280 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (15000 * ureg.feet,   2278 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (20000 * ureg.feet,   2020 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (25000 * ureg.feet,   1911 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (30000 * ureg.feet,   1818 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (35000 * ureg.feet,   1734 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
    (51000 * ureg.feet,    500 * ureg.feet / ureg.minute),  # active-climb median (n>=30/bin)
]),

descent_profile=VerticalProfile(points=[
    (    0 